# 04 Model Training & Evaluation — UAV-FD
Huấn luyện và đánh giá mô hình phân loại nhị phân (Normal = 0, Fault = 1)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    precision_score, 
    recall_score, 
    f1_score, 
    accuracy_score, 
    confusion_matrix
)

df_uav = pd.read_csv("../data/processed/imu_features.csv")

df_uav['Label'] = pd.to_numeric(df_uav['Label']).astype(int)

target = 'Label'
group_col = 'source_file'
metadata_cols = ['source_file', 'Window_ID', 'Label']
features = [col for col in df_uav.columns if col not in metadata_cols]

X = df_uav[features].values
y = df_uav[target].values
groups = df_uav[group_col].values

print(f"Tổng số cửa sổ đặc trưng (dòng): {len(df_uav):,}")
print(f"Số lượng đặc trưng (features): {len(features)}")
print(f"Tổng số chuyến bay (files): {df_uav[group_col].nunique()}")
print("\nSố lượng cửa sổ theo từng trạng thái (0 = Normal, 1 = Fault):")
print(df_uav[target].value_counts())

In [ ]:
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

fold_info = []
for fold_idx, (train_idx, test_idx) in enumerate(sgkf.split(X, y, groups=groups), start=1):
    train_files = np.unique(groups[train_idx])
    test_files = np.unique(groups[test_idx])
    
    # Kiểm tra rò rỉ dữ liệu giữa Train và Test (Data Leakage check)
    overlap = set(train_files).intersection(set(test_files))
    
    fold_info.append({
        'Fold': fold_idx,
        'Số mẫu Train': len(train_idx),
        'Số mẫu Test': len(test_idx),
        'Số file Train': len(train_files),
        'Số file Test': len(test_files),
        'Danh sách file Test': ", ".join(test_files),
        'Trùng file (Leakage)': len(overlap)
    })

df_folds = pd.DataFrame(fold_info)
print("=== BẢNG PHÂN CHIA 5-FOLD THEO CHUYẾN BAY (CHỐNG DATA LEAKAGE) ===")
display(df_folds)

In [ ]:
# 1. Khởi tạo mô hình với random_state=42 cố định
lr_model = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(random_state=42, max_iter=1000))
])

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    random_state=42,
    n_jobs=-1
)

lr_records = []
rf_records = []

cm_lr_total = np.zeros((2, 2), dtype=int)
cm_rf_total = np.zeros((2, 2), dtype=int)

# Lưu lại dự đoán để làm Error Analysis
all_test_indices = []
all_preds_lr = []
all_preds_rf = []

print("Đang huấn luyện 5 folds trên 36 đặc trưng cửa sổ...")

for fold_idx, (train_idx, test_idx) in enumerate(sgkf.split(X, y, groups=groups), start=1):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    
    # --- Mô hình 1: Logistic Regression ---
    lr_model.fit(X_train, y_train)
    y_pred_lr = lr_model.predict(X_test)
    
    lr_records.append({
        'Fold': fold_idx,
        'Precision': precision_score(y_test, y_pred_lr, pos_label=1, zero_division=0),
        'Recall': recall_score(y_test, y_pred_lr, pos_label=1, zero_division=0),
        'F1-score': f1_score(y_test, y_pred_lr, pos_label=1, zero_division=0),
        'Accuracy': accuracy_score(y_test, y_pred_lr)
    })
    cm_lr_total += confusion_matrix(y_test, y_pred_lr, labels=[0, 1])
    
    # --- Mô hình 2: Random Forest ---
    rf_model.fit(X_train, y_train)
    y_pred_rf = rf_model.predict(X_test)
    
    rf_records.append({
        'Fold': fold_idx,
        'Precision': precision_score(y_test, y_pred_rf, pos_label=1, zero_division=0),
        'Recall': recall_score(y_test, y_pred_rf, pos_label=1, zero_division=0),
        'F1-score': f1_score(y_test, y_pred_rf, pos_label=1, zero_division=0),
        'Accuracy': accuracy_score(y_test, y_pred_rf)
    })
    cm_rf_total += confusion_matrix(y_test, y_pred_rf, labels=[0, 1])
    
    all_test_indices.extend(test_idx)
    all_preds_lr.extend(y_pred_lr)
    all_preds_rf.extend(y_pred_rf)
    
    print(f" -> Đã chạy xong Fold {fold_idx}/5")

df_lr_folds = pd.DataFrame(lr_records)
df_rf_folds = pd.DataFrame(rf_records)

print("\n=== KẾT QUẢ TỪNG FOLD CỦA LOGISTIC REGRESSION ===")
display(df_lr_folds.round(4))

print("\n=== KẾT QUẢ TỪNG FOLD CỦA RANDOM FOREST ===")
display(df_rf_folds.round(4))

# Bảng tổng hợp Mean ± Std
summary_table = pd.DataFrame({
    'Metric (Lớp Fault = 1)': ['Precision', 'Recall', 'F1-score', 'Accuracy'],
    'Logistic Regression (Mean ± Std)': [
        f"{df_lr_folds['Precision'].mean():.4f} ± {df_lr_folds['Precision'].std():.4f}",
        f"{df_lr_folds['Recall'].mean():.4f} ± {df_lr_folds['Recall'].std():.4f}",
        f"{df_lr_folds['F1-score'].mean():.4f} ± {df_lr_folds['F1-score'].std():.4f}",
        f"{df_lr_folds['Accuracy'].mean():.4f} ± {df_lr_folds['Accuracy'].std():.4f}"
    ],
    'Random Forest (Mean ± Std)': [
        f"{df_rf_folds['Precision'].mean():.4f} ± {df_rf_folds['Precision'].std():.4f}",
        f"{df_rf_folds['Recall'].mean():.4f} ± {df_rf_folds['Recall'].std():.4f}",
        f"{df_rf_folds['F1-score'].mean():.4f} ± {df_rf_folds['F1-score'].std():.4f}",
        f"{df_rf_folds['Accuracy'].mean():.4f} ± {df_rf_folds['Accuracy'].std():.4f}"
    ]
})

print("\n=== BẢNG TỔNG HỢP SO SÁNH HIỆU NĂNG 2 MÔ HÌNH ===")
display(summary_table)

In [ ]:
# 1. Vẽ Confusion Matrix tổng hợp sau 5 Folds
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(cm_lr_total, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[0],
            xticklabels=['Dự đoán Normal (0)', 'Dự đoán Fault (1)'],
            yticklabels=['Thực tế Normal (0)', 'Thực tế Fault (1)'])
axes[0].set_title('Confusion Matrix - Logistic Regression (Window Features)')

sns.heatmap(cm_rf_total, annot=True, fmt='d', cmap='Greens', cbar=False, ax=axes[1],
            xticklabels=['Dự đoán Normal (0)', 'Dự đoán Fault (1)'],
            yticklabels=['Thực tế Normal (0)', 'Thực tế Fault (1)'])
axes[1].set_title('Confusion Matrix - Random Forest (Window Features)')

plt.suptitle("So sánh Confusion Matrix tổng hợp trên 5 Folds (36 đặc trưng cửa sổ)")
plt.tight_layout()
plt.show()

# 2. Thực hiện Statistical Test so sánh giữa Random Forest và Logistic Regression
stat_results = []
for metric in ['Precision', 'Recall', 'F1-score', 'Accuracy']:
    t_stat, p_ttest = stats.ttest_rel(df_rf_folds[metric], df_lr_folds[metric])
    try:
        w_stat, p_wilc = stats.wilcoxon(df_rf_folds[metric], df_lr_folds[metric], zero_method='zsplit')
    except ValueError:
        w_stat, p_wilc = np.nan, np.nan
        
    stat_results.append({
        'Metric': metric,
        'Chênh lệch Mean (RF - LR)': df_rf_folds[metric].mean() - df_lr_folds[metric].mean(),
        'Paired t-test (t-stat)': t_stat,
        'Paired t-test (p-value)': p_ttest,
        'Wilcoxon (p-value)': p_wilc
    })

df_stat_test = pd.DataFrame(stat_results)
print("=== KẾT QUẢ KIỂM ĐỊNH THỐNG KÊ SO SÁNH 2 MÔ HÌNH ===")
display(df_stat_test.round(6))

In [ ]:
# 1. Gộp kết quả dự đoán vào bảng để phân tích lỗi theo từng file
df_eval = df_uav.iloc[all_test_indices].copy()
df_eval['Pred_LR'] = all_preds_lr
df_eval['Pred_RF'] = all_preds_rf
df_eval['Error_LR'] = (df_eval['Label'] != df_eval['Pred_LR']).astype(int)
df_eval['Error_RF'] = (df_eval['Label'] != df_eval['Pred_RF']).astype(int)

error_by_file = df_eval.groupby(['source_file', 'Label']).agg(
    Tong_Cua_So=('Label', 'count'),
    Sai_LR=('Error_LR', 'sum'),
    Ty_le_sai_LR=('Error_LR', lambda x: round(x.mean() * 100, 2)),
    Sai_RF=('Error_RF', 'sum'),
    Ty_le_sai_RF=('Error_RF', lambda x: round(x.mean() * 100, 2))
).reset_index()

print("=== BẢNG PHÂN TÍCH LỖI (ERROR ANALYSIS) THEO TỪNG CHUYẾN BAY ===")
display(error_by_file)

# 2. Vẽ biểu đồ Top 15 Feature Importance của Random Forest
plt.figure(figsize=(10, 6))
importances = pd.Series(rf_model.feature_importances_, index=features).sort_values(ascending=True)
importances.tail(15).plot(kind='barh', color='teal')
plt.title('Top 15 Đặc trưng quan trọng nhất (Random Forest Feature Importance)')
plt.xlabel('Mức độ đóng góp (Importance Score)')
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

print("\nTop 15 đặc trưng đóng góp cao nhất:")
display(importances.sort_values(ascending=False).head(15).round(4))